In [1]:
import numpy as np
import matplotlib.pyplot as plt

def loss_stats(T, L):
    # Calculate the mean
    T_mean = np.mean(T)
    L_mean = np.mean(L)
    
    # Calculate the standard deviation above and below the mean for training_time
    T_above = T[T > T_mean] - T_mean
    T_below = T_mean - T[T < T_mean]
    
    T_std_above = np.std(np.concatenate((T_above, -T_above)))
    T_std_below = np.std(np.concatenate((T_below, -T_below)))
    
    # Calculate points above and below the mean for val_loss
    L_above = L[L > L_mean] - L_mean
    L_below = L_mean - L[L < L_mean]
    
    L_std_above = np.std(np.concatenate((L_above, -L_above)))
    L_std_below = np.std(np.concatenate((L_below, -L_below)))

    return {"T_mean": T_mean, "T_std_above": T_std_above, "T_std_below": T_std_below,
            "L_mean": L_mean, "L_std_above": L_std_above, "L_std_below": L_std_below}

def losses_stats(T, L):
    t_min = np.min(T[:,0])
    t_max = np.max(T[:,-1])
    common_time = np.linspace(t_min, t_max, 500)
    
    # Interpolate all losses onto common_time grid
    loss_interp = np.zeros((T.shape[0], len(common_time)))
    for i in range(T.shape[0]):
        loss_interp[i] = np.interp(common_time, T[i], L[i])
    
    mean_loss = np.mean(loss_interp, axis=0)
    
    # Separate deviations above and below the mean and flip for proper std
    diff = loss_interp - mean_loss  # shape: (No_Exp, len(common_time))
    
    std_above = np.zeros_like(mean_loss)
    std_below = np.zeros_like(mean_loss)
    
    for j in range(len(common_time)):
        above = diff[:, j][diff[:, j] > 0]
        below = diff[:, j][diff[:, j] < 0]
        
        if len(above) > 0:
            std_above[j] = np.std(np.concatenate((above, -above)))
        if len(below) > 0:
            std_below[j] = np.std(np.concatenate((below, -below)))
    return {"T": common_time, "L_mean": mean_loss, "L_std_above": std_above, "L_std_below": std_below}

In [6]:
ex_name = "ex1"

In [7]:
# Adam
Adam_data = np.load(f"raw_loss_data/{ex_name}_Adam.npz")
Adam_losses = Adam_data["val_losses"]
Adam_times = Adam_data["training_times"]

mean_min_loss = np.mean(np.min(Adam_losses, axis=1))     
Adam_losses_stats = losses_stats(Adam_times, Adam_losses)

np.savez(f"min_loss_stats/{ex_name}_Adam.npz", mean_min_loss)
np.savez(f"loss_profile_stats/{ex_name}.npz", **Adam_losses_stats)


In [8]:
# ARFF 
ARFF_data = np.load(f"raw_loss_data/{ex_name}_ARFF.npz")
ARFF_loss = ARFF_data["val_loss"]
ARFF_time = ARFF_data["training_time"]

ARFF_loss_stats = loss_stats(ARFF_time, ARFF_loss)
np.savez(f"min_loss_stats/{ex_name}_ARFF.npz", **ARFF_loss_stats)